In [1]:
# ==========================================
# Imports
# ==========================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from typing import Tuple, List, Any
from anova_module import FullSupportAnova
from anova_module import batch_shapley_values
import shap

In [2]:
# ==========================================
# Data Loading & Preprocessing
# ==========================================

def load_and_preprocess_data() -> Tuple[np.ndarray, np.ndarray, OrdinalEncoder, LabelEncoder]:
    """
    Downloads the Car Evaluation dataset from UCI, performs ordinal encoding
    on categorical features, and returns processed NumPy arrays.

    Returns
    -------
    X_encoded : np.ndarray
        The feature matrix with categorical variables encoded as integers.
    y_encoded : np.ndarray
        The target vector encoded as integers.
    feature_encoder : OrdinalEncoder
        The fitted encoder for the features (useful for inverse transformation).
    target_encoder : LabelEncoder
        The fitted encoder for the target labels.
    """
    # Direct URL to the raw dataset on the UCI repository
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/car/car.data"
    
    # Define column names (the .data file lacks a header)
    column_names = ['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'class']
    
    print(f"Downloading data from {url}...")
    df = pd.read_csv(url, names=column_names)
    
    print(f"Data loaded successfully. Shape: {df.shape}")
    
    # Separate Features (X) and Target (y)
    X_raw = df.iloc[:, :-1].values
    y_raw = df.iloc[:, -1].values
    
    # --- Encoding ---
    # The raw data consists of strings (e.g., "vhigh", "small").
    # We must convert these to numerical representations for the Random Forest.
    
    # 1. Feature Encoding
    # OrdinalEncoder transforms each categorical feature into integers (0, 1, 2...)
    feature_encoder = OrdinalEncoder()
    X_encoded = feature_encoder.fit_transform(X_raw)
    
    # 2. Target Encoding
    # LabelEncoder transforms class labels ("unacc", "acc", etc.) into integers.
    target_encoder = LabelEncoder()
    y_encoded = target_encoder.fit_transform(y_raw)
    
    return X_encoded, y_encoded, feature_encoder, target_encoder

# ==========================================
# Random Forest Training
# ==========================================

# 1. Data Loading & Preprocessing
# We assume load_and_preprocess_data() is defined as in the previous step
X, y, enc_X, enc_y = load_and_preprocess_data()

# Extract class names for legible reporting (e.g., 'unacc', 'vgood')
target_classes = enc_y.classes_ 

# 2. Experimental Setup: Stratified Train/Test Split
# We use a 70/30 split. Stratification is crucial here due to class imbalance.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# 3. Model Initialization: Random Forest
# Random Forests are robust baselines for tabular data with ordinal features.
clf = RandomForestClassifier(n_estimators=100, random_state=42)

print("Training model...")
clf.fit(X_train, y_train)

# 4. Inference
y_pred = clf.predict(X_test)

# 5. Performance Reporting
acc = accuracy_score(y_test, y_pred)

print("\n" + "="*40)
print(f"RESULTS (Accuracy: {acc:.4f})")
print("="*40)

# Detailed classification report
print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=target_classes))

Data loaded successfully. Shape: (1728, 7)
Training model...

RESULTS (Accuracy: 0.9672)

--- Classification Report ---
              precision    recall  f1-score   support

         acc       0.92      0.93      0.93       115
        good       0.90      0.90      0.90        21
       unacc       0.98      0.98      0.98       363
       vgood       1.00      0.95      0.97        20

    accuracy                           0.97       519
   macro avg       0.95      0.94      0.95       519
weighted avg       0.97      0.97      0.97       519



In [3]:
%%time
# ==========================================
# Functional ANOVA Decomposition
# ==========================================

d = X.shape[1] # dimension
N = np.array([ X[: , j].max() + 1 for j in range(d) ]) # list of categories
r = int(np.prod(N)) # full dimension
P = 1/r * np.ones( r ) # vector of probabilities

def f_1(x): # class 1
    return(clf.predict_proba(x)[:,0])

def f_2(x): # class 2
    return(clf.predict_proba(x)[:,1])

def f_3(x): # class 3
    return(clf.predict_proba(x)[:,2])

def f_4(x): # class 4
    return(clf.predict_proba(x)[:,3])

F = [f_1 , f_2 , f_3 , f_4] # list of functions for each class


anova_shap = [] # list of generalized shapley values based on functional anova
for f_model in F:
    A = FullSupportAnova(N , P , f_model)
    S , Matrix = A.get_anova_full() # sets and f_A(X_A)
    shap_i = batch_shapley_values(d , S , Matrix) # generalized shapley values matrix for all obs
    anova_shap.append(shap_i)

CPU times: user 421 ms, sys: 28.2 ms, total: 449 ms
Wall time: 450 ms


In [4]:
# ==========================================
# KernelSHAP
# ==========================================

n_sample_background = 200
background = X[:n_sample_background]

# KernelSHAP
def kernel_shap(f , X_explain):
    explainer = shap.KernelExplainer(f , background)
    shap_values = explainer.shap_values(X_explain)
    return(shap_values)

In [5]:
%%time
# ==========================================
# KernelSHAP
# ==========================================

number = r - 1
X_explain = A._generate_tuples()[:number]

all_kernel_shap = []
for g in F:
    kernel_shap_g = kernel_shap(g , X_explain)
    all_kernel_shap.append(kernel_shap_g)

Using 200 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


  0%|          | 0/1727 [00:00<?, ?it/s]

Using 200 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


  0%|          | 0/1727 [00:00<?, ?it/s]

Using 200 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


  0%|          | 0/1727 [00:00<?, ?it/s]

Using 200 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


  0%|          | 0/1727 [00:00<?, ?it/s]

CPU times: user 1min 39s, sys: 2.45 s, total: 1min 42s
Wall time: 1min 42s


In [ ]:
# ==========================================
# Table of ISE
# ==========================================

P_red = 1/number * np.ones(number)

np.array([np.sum(((anova_shap[i][:number , :] - all_kernel_shap[i])**2).T * P_red , axis=1) for i in range(4)])

array([[2.01123261e-02, 1.29533867e-02, 9.62057670e-05, 2.75037594e-03,
        2.50105313e-04, 2.98341343e-03],
       [1.58470991e-03, 1.46948070e-03, 2.69347934e-05, 4.09998114e-04,
        1.18857332e-04, 5.11330245e-04],
       [2.89357167e-02, 1.45756384e-02, 1.70867649e-04, 7.08700651e-03,
        4.56296966e-04, 7.07322515e-03],
       [2.07139881e-03, 8.71936191e-04, 6.46691690e-05, 3.68443860e-04,
        3.97373917e-04, 1.20824268e-03]])